In [22]:
import pandas as pd
import numpy as np
import os
import sys
%load_ext autoreload
%autoreload 2

from apnea_detection_functions import apply_apnea_prediction_models

sys.path.append('/home/wolfgang/repos/ICU-Sleep/code1')
from airgo_features import compute_airgo_features

sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import *
%matplotlib widget
import matplotlib.pyplot as plt

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
data_dir = f'/media/ssd2/icu_final_files'

files = os.listdir(data_dir)
files.sort()


file = files[50]
study_id = file[-6:-3]

input_file_path = os.path.join(data_dir, file)

data, hdr, fs  = read_in_routine(input_file_path, check_airgo_scaling=False)

original_airgo_signals = ['band', 'accx', 'accy', 'accz']
signals_to_keep = original_airgo_signals
signals_to_keep = [x for x in signals_to_keep if x in data.columns]
data = data[signals_to_keep]


In [3]:
data = data.dropna(subset=['band'])

In [4]:
data2 = data.copy()

In [37]:
data = data2.iloc[fs*3600*2 : fs*3600*3,:]

In [39]:
data.to_csv('airgo_1h_sample.csv', index=True)

In [40]:
data

,band,accx,accy,accz
2019-05-21 19:16:02.600,2160.0,-0.319092,0.227417,-0.884766
2019-05-21 19:16:02.700,2160.0,-0.331299,0.232300,-0.895020
2019-05-21 19:16:02.800,2160.0,-0.324219,0.229370,-0.888672
2019-05-21 19:16:02.900,2156.0,-0.329346,0.225342,-0.891113
2019-05-21 19:16:03.000,2162.0,-0.343506,0.223145,-0.880859
...,...,...,...,...
2019-05-21 20:16:02.100,1979.0,-0.133545,0.054016,-0.967773
2019-05-21 20:16:02.200,1976.0,-0.121338,0.033661,-0.964355
2019-05-21 20:16:02.300,1969.0,-0.117249,0.043823,-0.974609
2019-05-21 20:16:02.400,1970.0,-0.124329,0.031616,-0.971680


In [25]:
data = airgo_resample_preprocess(data)

data = compute_airgo_features(data, fs=fs, complexity_features=1)

In [26]:
data.head()

,band,accx,accy,accz,band_unscaled,movavg_0_5s,movavg_1_2s,movavg_1min,deriv1,ventilation0,...,instability_index_30sec,ventilation_cvar_1min,IBI_cvar_1min,instability_index_1min,ventilation_cvar_2min,IBI_cvar_2min,instability_index_2min,ventilation_cvar_5min,IBI_cvar_5min,instability_index_5min
2019-05-21 19:16:02.600,0.522859,-0.032527,0.023182,-0.090190,2160.0,0.522859,0.516453,0.629159,0.000000,0.000000,...,0.222041,0.316455,0.126678,0.221567,0.271966,0.150015,0.210990,0.198062,0.203435,0.200748
2019-05-21 19:16:02.700,0.522859,-0.033772,0.023680,-0.091235,2160.0,0.516453,0.506386,0.629146,-0.006406,0.000000,...,0.221832,0.317285,0.126645,0.221965,0.271750,0.149880,0.210815,0.198038,0.203370,0.200704
2019-05-21 19:16:02.800,0.522859,-0.033050,0.023381,-0.090588,2160.0,0.520296,0.494031,0.629091,0.003844,0.003844,...,0.221601,0.318119,0.126617,0.222368,0.271537,0.149747,0.210642,0.198015,0.203305,0.200660
2019-05-21 19:16:02.900,0.497234,-0.033572,0.022971,-0.090837,2156.0,0.515171,0.485845,0.629079,-0.005125,0.000000,...,0.221350,0.318959,0.126594,0.222777,0.271328,0.149615,0.210471,0.197993,0.203243,0.200618
2019-05-21 19:16:03.000,0.535671,-0.035016,0.022747,-0.089792,2162.0,0.499797,0.476734,0.629025,-0.015375,0.000000,...,0.221083,0.319808,0.126576,0.223192,0.271121,0.149485,0.210303,0.197972,0.203181,0.200577


In [27]:
do_apply_apnea_prediction_models = True

if do_apply_apnea_prediction_models:
    data = apply_apnea_prediction_models(data, fs=fs)

Following Apnea Models loaded: ['rsr_a3', 'ro_a3']
rsr_a3 not applied, no SpO2 available.


Note: The function tries to run all possible apnea models. There is a version that works with AirGo signal only, and one that takes airgo+spo2 as input. One post-processing prediction column gets computed too which only makes sense if spo2 is available, and it is suitable for ICU Sleep Study only I think, so you can ignore this column.

Model is applied to 1-second-resolution, and it gives both the probability of an apnea (not sure if you need it) and the binarized output Apnea Yes/No.



In [28]:
data.apnea_pred_ro_a3.dropna().head(3)

2019-05-21 19:16:02.600    0.0
2019-05-21 19:16:03.600    0.0
2019-05-21 19:16:04.600    0.0
Name: apnea_pred_ro_a3, dtype: float64

In [32]:
print(f'Number of predicted Apnea/Hypopnea events: {(data.apnea_pred_ro_a3.dropna().diff() == 1).sum()}')

Number of predicted Apnea/Hypopnea events: 6


In [36]:
print('Apnea Predictions:')
data.apnea_pred_ro_a3.dropna()[data.apnea_pred_ro_a3.dropna().diff() == 1]

Apnea Predictions:


2019-05-21 19:31:16.600    1.0
2019-05-21 19:33:35.600    1.0
2019-05-21 19:34:56.600    1.0
2019-05-21 19:38:33.600    1.0
2019-05-21 19:41:26.600    1.0
2019-05-21 19:47:07.600    1.0
Name: apnea_pred_ro_a3, dtype: float64

In [30]:
fig, ax = plt.subplots(2,1, figsize = (8,6), sharex = True)
ax[0].plot(data.band_unscaled)
ax[1].plot(data.apnea_prob_ro_a3.dropna(), color='orange')
ax[1].plot(data.apnea_pred_ro_a3.dropna(), color = 'red')
ax[1].legend(['Probability', 'Prediction'])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …